# 02 · Model experiments — Trade Perfect
Same split (80/20 stratified, seed 42), same 17 features, encoders fitted on the training split only. Threshold rule: lowest threshold with **FPR < 5% and recall ≥ 85%**.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
DATA = ROOT / "data"
from app.engine import EngineConfig, load_historical_csv
from app.engine.reference import load_reference_data, compute_counterparty_tiers
plt.rcParams.update({"figure.figsize": (8, 3.6), "axes.spines.top": False, "axes.spines.right": False})
hist = load_historical_csv(str(DATA / "Historical_Trade_Dataset_50K.csv"))
refs = load_reference_data(str(DATA / "Ref_Data"), hist)
y = hist.settlement_failed
print(f"{len(hist):,} historical trades | fail rate {y.mean():.1%} | {hist.trade_date.min()} -> {hist.trade_date.max()}")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from app.engine.rules import evaluate_rules
from app.engine.features import fit_encoders, build_features, FEATURE_LIST
from app.engine.evaluate import compute_metrics, select_threshold, threshold_curve
yv = y.astype(int).to_numpy()
tr, te = train_test_split(np.arange(len(hist)), test_size=0.2, stratify=yv, random_state=42)
rule_ml = evaluate_rules(hist, refs, EngineConfig(), None).rule_score_ml
enc = fit_encoders(hist.iloc[tr], 100)
Xtr = build_features(hist.iloc[tr], rule_ml[tr], enc, refs); Xte = build_features(hist.iloc[te], rule_ml[te], enc, refs)
print(Xtr.shape, Xte.shape)

## 1. Compare models

In [ ]:
RF = dict(n_estimators=200, max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)
models = {
    "Random Forest (production)": RandomForestClassifier(**RF),
    "RF without fail-rate features": "noTE",
    "RF class_weight=None": RandomForestClassifier(**{**RF, "class_weight": None}),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=42),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "MLP neural net (64-32)": make_pipeline(StandardScaler(), MLPClassifier((64, 32), max_iter=300, early_stopping=True, random_state=42)),
}
TE = ["Counterparty_Fail_Rate", "Symbol_Fail_Rate", "CP_Symbol_Fail_Rate"]
rows, probs = [], {}
for name, m in models.items():
    a, b = (Xtr.drop(columns=TE), Xte.drop(columns=TE)) if m == "noTE" else (Xtr, Xte)
    m = RandomForestClassifier(**RF) if m == "noTE" else m
    p = m.fit(a, yv[tr]).predict_proba(b)[:, 1]; probs[name] = p
    t, ok = select_threshold(yv[te], p)
    rows.append({"model": name, "threshold": t, **{k: v for k, v in compute_metrics(yv[te], p, t).items() if k not in ("tp", "fp", "tn", "fn")}, "all 6 met": ok and compute_metrics(yv[te], p, t)["accuracy"] > 0.90})
p_rule = Xte["Rule_Engine_Output"].to_numpy(); t, _ = select_threshold(yv[te], p_rule)
rows.append({"model": "Rules alone (no ML)", "threshold": t, **{k: v for k, v in compute_metrics(yv[te], p_rule, t).items() if k not in ("tp", "fp", "tn", "fn")}, "all 6 met": False})
res = pd.DataFrame(rows).set_index("model"); res

Every model lands at AUC ≈ 0.959 → that is the **ceiling of this data**. The Random Forest meets all six targets and supports fast, exact TreeSHAP → production model. The neural net is slightly worse and harder to explain.

## 2. The threshold trade-off (production RF)
Raising the threshold lowers false alarms (FPR) but also recall. All six targets hold only in a narrow band (~0.79–0.81).

In [ ]:
curve = pd.DataFrame(threshold_curve(yv[te], probs["Random Forest (production)"])).set_index("threshold")
ax = curve[["false_positive_rate", "recall", "accuracy"]].plot(marker=".", title="Metrics vs decision threshold")
ax.axhline(0.05, ls="--", c="#f5222d", lw=1); ax.axhline(0.85, ls="--", c="#52c41a", lw=1); ax.axhline(0.90, ls="--", c="#1677ff", lw=1)
ax.axvline(0.80, c="k", lw=1); ax.set_ylim(0, 1); plt.show()
curve.loc[[0.75, 0.775, 0.8, 0.825, 0.85], ["recall", "accuracy", "false_positive_rate"]]

## 3. Production model v1 — the saved report

In [ ]:
rep = json.loads((ROOT / "backend" / "ml_artifacts" / "model_v1_report.json").read_text())
print("threshold", rep["decision_threshold"]); display(pd.Series(rep["metrics"]).to_frame("v1"))
cm = rep["diagnostics"]["confusion_matrix"]
print(f"Confusion matrix  TP {cm['tp']}  FP {cm['fp']}  TN {cm['tn']}  FN {cm['fn']}")
print("5-fold CV estimate:", {k: rep["diagnostics"]["cv_estimate"]["holdout_metrics_at_cv_threshold"][k] for k in ("accuracy", "recall", "false_positive_rate")})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
g = pd.DataFrame(rep["global_shap"]).set_index("feature").mean_abs_shap.sort_values()
g.plot.barh(ax=axes[0], color="#1677ff", title="Global feature importance (mean |SHAP|)")
m = pd.DataFrame(rep["monthly_evaluations"]).set_index("dataset_label")
m.index = m.index.str[6:]
m[["accuracy", "recall", "precision", "false_positive_rate"]].plot(ax=axes[1], marker="o", title="Monthly evaluation (holdout)")
axes[1].axhline(0.90, ls="--", c="#1677ff", lw=1); axes[1].axhline(0.05, ls="--", c="#f5222d", lw=1)
plt.tight_layout(); plt.show()

## 4. Data drift: 20K demo file vs training baseline (PSI)
GREEN < 0.1 ≤ AMBER ≤ 0.2 < RED

In [ ]:
from app.engine import Predictor, compute_drift
from app.engine.normalize import read_trade_csv, normalize_records
from app.engine.hard_gates import run_hard_gates
from app.engine.pipeline import _accepted_frame
from datetime import date
pred = Predictor.load(str(ROOT / "backend" / "ml_artifacts" / "model_v1.joblib"))
b = normalize_records(read_trade_csv(str(DATA / "Trade_Dataset_20K.csv"))); gg = run_hard_gates(b, refs)
df = _accepted_frame(b, gg, [i for i, e in enumerate(gg.errors) if not e])
X = pred.build_features(df, evaluate_rules(df, refs, EngineConfig(), date(2026, 9, 28)).rule_score_ml, refs)
d = pd.DataFrame(compute_drift(rep["baseline_stats"], X)).set_index("feature")
ax = d.psi.sort_values().plot.barh(color=d.psi.sort_values().map(lambda v: "#f5222d" if v > 0.2 else "#faad14" if v >= 0.1 else "#52c41a"), figsize=(8, 5), title="PSI per feature")
ax.axvline(0.1, ls="--", c="#faad14"); ax.axvline(0.2, ls="--", c="#f5222d"); plt.show(); d

## Conclusions
- Production: Random Forest (200 trees, depth 12, leaf 5, balanced), smoothing k=100, threshold 0.80 → AUC 0.959, precision 0.973, recall 0.879, F1 0.924, accuracy 0.903, FPR 4.96% (all six targets).
- Honest check: 5-fold CV estimate puts FPR at ~5.1% — the data has a floor (≈5% of non-failing trades have SSI = NO).
- Rules alone cannot meet both targets: at recall ≥ 85% their FPR is 7.2%; pushing FPR under 5% drops recall to 80%. The hybrid model meets both.
- Drift: the demo file differs strongly in settlement timing and counterparty mix (RED); SSI/currency stable (GREEN).